# Análisis Exploratorio de los datos de vino

Este notebook carga los archivos `winequality-red.csv` y `winequality-white.csv` para explorar las características y la calidad de los vinos.

## 0. Información del dataset

El archivo `winequality.names` contiene la descripción del conjunto de datos. A continuación, se muestra su contenido:

In [ ]:
with open('winequality.names', 'r') as f:
    content = f.read()
    print(content)

## 2. Cargar librerías y datos

Importamos `pandas` y cargamos los dos archivos CSV.

In [ ]:
import pandas as pd

red = pd.read_csv('winequality-red.csv', sep=';')
white = pd.read_csv('winequality-white.csv', sep=';')

red.shape, white.shape

### 2.1 Validación y limpieza de los datos

Revisamos valores faltantes, duplicados y los tipos de datos antes de continuar con el análisis.

In [ ]:
for df, name in [(red, 'red'), (white, 'white')]:
    print(name.upper())
    print(df.isna().sum())
    print('duplicados:', df.duplicated().sum())
    print(df.dtypes)
    print('-'*40)


In [ ]:
red['wine_type'] = 'red'
white['wine_type'] = 'white'
wine = pd.concat([red, white], ignore_index=True)
wine['quality_label'] = pd.cut(wine['quality'], bins=[2,5,7,10], labels=['bajo','medio','alto'])
wine.shape
wine['wine_type'].value_counts()


## 3. Exploración inicial

Revisamos las primeras filas, los tipos de datos y estadísticas generales.

In [ ]:
red.head()

In [ ]:
white.head()

In [ ]:
red.info()
print()
white.info()

In [ ]:
red.describe()

In [ ]:
white.describe()

### 4. Estadísticas por tipo de vino

Miramos las diferencias generales entre vino tinto y blanco usando estadísticas agrupadas.

In [ ]:
wine.groupby('wine_type').describe().T


### 4.1 Análisis por tipo de vino

Exploramos cómo las características difieren entre vino tinto y blanco usando técnicas de EDA.

In [ ]:
wine = pd.concat([red, white], ignore_index=True)
wine['wine_type'] = ['red'] * len(red) + ['white'] * len(white)

print("Conteo por tipo de vino:\n", wine["wine_type"].value_counts())
print("\nCalidad promedio por tipo de vino:\n", wine.groupby("wine_type")["quality"].mean())
print("\nEstadísticas de alcohol y pH por tipo de vino:\n", wine.groupby("wine_type")[['alcohol', 'pH']].agg(["mean", "median", "std"]))

fig, axs = plt.subplots(1, 2, figsize=(12, 5), tight_layout=True)
sns.boxplot(x="wine_type", y="alcohol", data=wine, ax=axs[0])
axs[0].set_title("Alcohol por tipo de vino")
sns.boxplot(x="wine_type", y="pH", data=wine, ax=axs[1])
axs[1].set_title("pH por tipo de vino")
plt.show()

fig, ax = plt.subplots(1, 2, figsize=(12, 5), tight_layout=True)
sns.histplot(data=wine, x="quality", hue="wine_type", multiple="dodge", shrink=0.8, ax=ax[0])
ax[0].set_title("Distribución de calidad por tipo de vino")
sns.scatterplot(x="alcohol", y="quality", hue="wine_type", data=wine, alpha=0.6, ax=ax[1])
ax[1].set_title("Alcohol vs calidad por tipo de vino")
plt.show()

## 4.2 Distribución de la calidad

Comparamos la frecuencia de las puntuaciones de calidad para vino tinto y vino blanco.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 4), tight_layout=True)
red['quality'].value_counts().sort_index().plot(kind='bar', ax=ax[0], color='darkred', title='Calidad Vino Tinto')
white['quality'].value_counts().sort_index().plot(kind='bar', ax=ax[1], color='goldenrod', title='Calidad Vino Blanco')
ax[0].set_xlabel('Calidad')
ax[0].set_ylabel('Cantidad')
ax[1].set_xlabel('Calidad')
ax[1].set_ylabel('Cantidad')
plt.show()

## 5. Correlaciones con la calidad

Calculamos la correlación de cada variable con la calidad para entender qué factores están relacionados.

In [ ]:
corr_red = red.corr()['quality'].sort_values(ascending=False)
corr_white = white.corr()['quality'].sort_values(ascending=False)
pd.DataFrame({'red_quality_corr': corr_red, 'white_quality_corr': corr_white})

### 5.1 Visualizaciones adicionales y nueva variable

Exploramos relaciones clave con gráficos y creamos una variable de apoyo para el análisis de calidad.

In [ ]:
import seaborn as sns
sns.set(style="whitegrid")
fig, axs = plt.subplots(2, 2, figsize=(14, 10), tight_layout=True)
sns.boxplot(x="wine_type", y="alcohol", data=wine, ax=axs[0,0])
axs[0,0].set_title("Alcohol por tipo de vino")
sns.scatterplot(x="alcohol", y="quality", hue="wine_type", data=wine, ax=axs[0,1])
axs[0,1].set_title("Alcohol vs calidad")
sns.histplot(data=wine, x="pH", hue="wine_type", kde=True, ax=axs[1,0])
axs[1,0].set_title("Distribución de pH")
sns.boxplot(x="wine_type", y="residual sugar", data=wine, ax=axs[1,1])
axs[1,1].set_title("Residual sugar por tipo de vino")
plt.show()
plt.figure(figsize=(10, 8))
sns.heatmap(wine.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Mapa de calor de correlaciones")
plt.show()


### 6. Ingeniería de variables

Creamos una nueva variable de relación de acidez y una bandera de alcohol alto para ver si agregan información útil.

In [ ]:
wine["acidity_ratio"] = wine["fixed acidity"] / (wine["volatile acidity"] + 0.01)
wine["high_alcohol"] = wine["alcohol"] > wine["alcohol"].median()
wine[["acidity_ratio", "high_alcohol", "quality"]].head()
wine[["acidity_ratio", "quality"]].corr()


## 6.1 Comparación de características clave

Vemos cómo cambian las variables principales entre los vinos tintos y blancos.

In [ ]:
summary = pd.DataFrame({
    'red_mean': red.mean(),
    'white_mean': white.mean(),
    'red_std': red.std(),
    'white_std': white.std()
})
summary.loc[['alcohol', 'pH', 'residual sugar', 'citric acid', 'sulphates', 'volatile acidity']]

## 7. Conclusiones iniciales

Añade aquí observaciones sobre la diferencia entre vinos tintos y blancos, los factores de calidad y qué análisis adicional deseas realizar.